## EVACUATION GRAPH

## 1. Import the needed libraries

In [1]:
from topologicpy.Vertex import Vertex
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Graph import Graph
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Color import Color

c:\Users\cutro\miniconda3\envs\graphml\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Check the TopologicPy version

In [2]:
print(Helper.Version())

The version that you are using (0.9.21) is EQUAL TO the latest version available on PyPI.


## 3. Set renderer

In [3]:
renderer = "vscode"

## 4. Import OBJ file

In [4]:
objects = Topology.ByOBJPath(r"C:\Users\cutro\Documents\GitHub\macad\graphml\Assets\Layout_Rooms.obj")
print("Objects: ", objects) #we obtain one cluser per room

Objects:  [<topologic_core.Cluster object at 0x000001F66B0773F0>, <topologic_core.Cluster object at 0x000001F61B7840F0>, <topologic_core.Cluster object at 0x000001F61B784570>, <topologic_core.Cluster object at 0x000001F61B826630>, <topologic_core.Cluster object at 0x000001F61B826330>, <topologic_core.Cluster object at 0x000001F63C666070>, <topologic_core.Cluster object at 0x000001F67F4F75B0>, <topologic_core.Cluster object at 0x000001F66B3D7270>, <topologic_core.Cluster object at 0x000001F66AEF67F0>, <topologic_core.Cluster object at 0x000001F67B42D4B0>, <topologic_core.Cluster object at 0x000001F66AFCBB30>, <topologic_core.Cluster object at 0x000001F61B3DA0B0>, <topologic_core.Cluster object at 0x000001F67FC3E8F0>, <topologic_core.Cluster object at 0x000001F66AFDFC30>, <topologic_core.Cluster object at 0x000001F61B2D8630>, <topologic_core.Cluster object at 0x000001F67F9B94B0>, <topologic_core.Cluster object at 0x000001F67F8F02B0>, <topologic_core.Cluster object at 0x000001F63C623930>]

Use selectors to identify different rooms by color, using the name set for each room in Rhino

In [5]:
cells = []
selectors = [] #vertices inside the room that have dictionaries in them, used to select the rooms based on the name added in Rhino
for object in objects:
    d = Topology.Dictionary(object)
    faces = Topology.Faces(object)
    if len(faces) > 1:
        c = Cell.ByFaces(faces)
        c = Topology.RemoveCollinearEdges(c) #remove any two edges that are collinear
        s = Topology.InternalVertex(c) #garantee to have a vertex inside the cell
        name = Dictionary.ValueAtKey(d, "name")
        if "Classroom" in name:
            color = "red"
        elif "Office" in name:
            color = "orange"
        elif "Corridor" in name:
            color = "green"
        elif "Stair" in name:
            color = "blue"
        elif "Bathroom" in name:
            color = "violet"
        else:
            color = "cyan"
        d = Dictionary.SetValuesAtKeys(d, ["color", "vertex_size"], [color, 15]) #add some extra values (color and vertex size) to the dictionary for later use in visualization
        s = Topology.SetDictionary(s, d) #set the dictionary of s to be d
        selectors.append(s)
        cells.append(c) 
        print(Dictionary.Keys(d), Dictionary.Values(d))

print("Number of cells: ", len(cells))

['color', 'group', 'material', 'name', 'opacity', 'vertex_size'] ['cyan', 'Lobby_1', '', 'Lobby_1', 1.0, 15]
['color', 'group', 'material', 'name', 'opacity', 'vertex_size'] ['green', 'Corridor_1', '', 'Corridor_1', 1.0, 15]
['color', 'group', 'material', 'name', 'opacity', 'vertex_size'] ['green', 'Corridor_2', '', 'Corridor_2', 1.0, 15]
['color', 'group', 'material', 'name', 'opacity', 'vertex_size'] ['violet', 'Bathroom_1', '', 'Bathroom_1', 1.0, 15]
['color', 'group', 'material', 'name', 'opacity', 'vertex_size'] ['violet', 'Bathroom_3', '', 'Bathroom_3', 1.0, 15]
['color', 'group', 'material', 'name', 'opacity', 'vertex_size'] ['violet', 'Bathroom_2', '', 'Bathroom_2', 1.0, 15]
['color', 'group', 'material', 'name', 'opacity', 'vertex_size'] ['orange', 'Office_1', '', 'Office_1', 1.0, 15]
['color', 'group', 'material', 'name', 'opacity', 'vertex_size'] ['orange', 'Office_2', '', 'Office_2', 1.0, 15]
['color', 'group', 'material', 'name', 'opacity', 'vertex_size'] ['orange', 'Offic

In [11]:
cc = CellComplex.ByCells(cells)
cc = Topology.RemoveCoplanarFaces(cc) #to remove the triangulation of the cells
cc = Topology.TransferDictionariesBySelectors(cc, selectors, tranCells=True) #here we use the selectors, we transfer teh selectors only to cell and not to faces
cc_cells = Topology.Cells(cc) #to check that the dictionaries have been transfered to the cells correctly
for cc_cell in cc_cells:
    d = Topology.Dictionary(cc_cell)
    print("Cell dictionary: ", Dictionary.Keys(d), Dictionary.Values(d))
Topology.Analyze(cc)

Cell dictionary:  ['aabb', 'color', 'group', 'material', 'name', 'opacity', 'vertex_size'] [[20.0, -3.0, 0.0, 30.0, 8.0, 3.5], 'cyan', 'Lobby_1', '', 'Lobby_1', 1.0, 15]
Cell dictionary:  ['aabb', 'color', 'group', 'material', 'name', 'opacity', 'vertex_size'] [[30.0, -5.0, 0.0, 37.0, 0.0, 3.5], 'red', 'Classroom_1', '', 'Classroom_1', 1.0, 15]
Cell dictionary:  ['aabb', 'color', 'group', 'material', 'name', 'opacity', 'vertex_size'] [[30.0, 0.0, 0.0, 61.0, 2.0, 3.5], 'green', 'Corridor_2', '', 'Corridor_2', 1.0, 15]
Cell dictionary:  ['aabb', 'color', 'group', 'material', 'name', 'opacity', 'vertex_size'] [[25.0, 2.0, 0.0, 30.0, 8.0, 3.5], 'blue', 'Stair_2', '', 'Stair_2', 1.0, 15]
Cell dictionary:  ['aabb', 'color', 'group', 'material', 'name', 'opacity', 'vertex_size'] [[12.0, 2.0, 0.0, 20.0, 8.0, 3.5], 'orange', 'Office_3', '', 'Office_3', 1.0, 15]
Cell dictionary:  ['aabb', 'color', 'group', 'material', 'name', 'opacity', 'vertex_size'] [[4.0, 0.0, 0.0, 20.0, 2.0, 3.5], 'green', '

'OVERALL ANALYSIS\n================\nThe shape is a cellComplex.\nNumber of cells = 18\nNumber of shells = 18\nNumber of faces = 98\nNumber of wires = 98\nNumber of edges = 169\nNumber of vertices = 90\n\n\nINDIVIDUAL ANALYSIS\n================\nThe shape is a cellComplex.\nNumber of cells = 18\n================\n  The shape is a cell.\n  Number of shells = 1\n  ================\n    The shape is a shell.\n    Number of faces = 11\n    ================\n      The shape is a face.\n      Number of wires = 1\n      ================\n        The shape is a wire.\n        Number of edges = 9\n        ================\n          The shape is an edge.\n          Number of vertices = 2\n          ================\n            The shape is a vertex.\n            ================\n            The shape is a vertex.\n            ================\n          The shape is an edge.\n          Number of vertices = 2\n          ================\n            The shape is a vertex.\n            ========

## 5. Show the geometry

In [12]:
Topology.Show(cc,
              backgroundColor="white",
              width=500,
              height=500,
              renderer=renderer)

In [13]:
Topology.Show(cc_cells,
              backgroundColor="white",
              width=500,
              height=500,
              faceColorKey="color",
              faceOpacity=1,
              renderer=renderer)

## 6. Derive Dual Graph

In [14]:
g1 = Graph.ByTopology(cc) #direct means that we are ignoring the doors

## 7. Visualization properties and Show Geometry and Graph

In [17]:
vertices = Graph.Vertices(g1)
for v in vertices:
    d = Topology.Dictionary(v)
    print("Vertex dictionary: ", Dictionary.Keys(d), Dictionary.Values(d)) #the dictionary of the vertices have the color property set for the room that they represent
edges = Graph.Edges(g1)
for e in edges:
    d = Topology.Dictionary(e)
    d = Dictionary.SetValuesAtKeys(d, ["width", "color"], [4, "black"]) #to expand the dictionary and not overwrite it and then put it back
    e = Topology.SetDictionary(e, d)

Vertex dictionary:  ['aabb', 'category', 'color', 'group', 'material', 'name', 'opacity', 'vertex_size'] [[61.0, 0.0, 0.0, 65.0, 8.0, 3.5], 0, 'blue', 'Stair_3', '', 'Stair_3', 1.0, 15]
Vertex dictionary:  ['aabb', 'category', 'color', 'group', 'material', 'name', 'opacity', 'vertex_size'] [[44.0, -5.0, 0.0, 51.0, 0.0, 3.5], 0, 'red', 'Classroom_3', '', 'Classroom_3', 1.0, 15]
Vertex dictionary:  ['aabb', 'category', 'color', 'group', 'material', 'name', 'opacity', 'vertex_size'] [[44.0, 2.0, 0.0, 52.0, 8.0, 3.5], 0, 'violet', 'Bathroom_2', '', 'Bathroom_2', 1.0, 15]
Vertex dictionary:  ['aabb', 'category', 'color', 'group', 'material', 'name', 'opacity', 'vertex_size'] [[12.0, 2.0, 0.0, 20.0, 8.0, 3.5], 0, 'orange', 'Office_3', '', 'Office_3', 1.0, 15]
Vertex dictionary:  ['aabb', 'category', 'color', 'group', 'material', 'name', 'opacity', 'vertex_size'] [[30.0, 2.0, 0.0, 38.0, 8.0, 3.5], 0, 'orange', 'Office_4', '', 'Office_4', 1.0, 15]
Vertex dictionary:  ['aabb', 'category', 'colo

In [18]:
Topology.Show(cc, g1,
                vertexSizeKey="vertex_size",
                vertexColorKey="color",
                edgeWidthKey="width",
                edgeColorKey="color",
                faceOpacity=0.3,
                backgroundColor="white",
                width=500,
                height=500,
                renderer=renderer)

In [19]:
vertices = Graph.Vertices(g1)
for v in vertices:
    d = Dictionary.ByKeysValues(["size", "color"], [18, "red"])
    v = Topology.SetDictionary(v, d)
edges = Graph.Edges(g1)
for e in edges:
    d = Topology.Dictionary(e)
    d = Dictionary.SetValuesAtKeys(d, ["width", "color"], [4, "black"]) #to expand the dictionary and not overwrite it and then put it back
    e = Topology.SetDictionary(e, d)

In [20]:
Topology.Show(cc, g1,
              vertexSizeKey="size",
              vertexColorKey="color",
              edgeWidthKey="width",
              edgeColorKey="color",
              faceOpacity=0.3,
              backgroundColor="white",
              width=500,
              height=500,
              renderer=renderer)

## Add Apertures from OBJ

In [21]:
apertures = []
objects = Topology.ByOBJPath(r"C:\Users\cutro\Documents\GitHub\macad\graphml\Assets\Layout_Test.obj", selfMerge=True)
print("Objects: ", objects)
#faces = [Topology.Faces(object)[0] for object in objects] #list comprehension to get the faces of each object
for face in faces:
    #face = Topology.RemoveCollinearEdges(faces)
    d = Dictionary.ByKeysValues(["type", "color", "vertx_size"], ["door", "brown", 20])
    face = Topology.SetDictionary(face, d)
    apertures.append(face)
print("Apertures: ", apertures)
print("Number of apertures: ", len(apertures))

Objects:  [<topologic_core.Face object at 0x000001F621A89630>, <topologic_core.Face object at 0x000001F621A899F0>, <topologic_core.Face object at 0x000001F621A89CF0>, <topologic_core.Face object at 0x000001F621A897B0>, <topologic_core.Face object at 0x000001F621A8B630>, <topologic_core.Face object at 0x000001F621A888B0>, <topologic_core.Face object at 0x000001F621A89770>, <topologic_core.Face object at 0x000001F621A8ADF0>, <topologic_core.Face object at 0x000001F621A887B0>, <topologic_core.Face object at 0x000001F621A899B0>, <topologic_core.Face object at 0x000001F621A889B0>, <topologic_core.Face object at 0x000001F621A898B0>, <topologic_core.Face object at 0x000001F621A8AAB0>, <topologic_core.Face object at 0x000001F621A89830>, <topologic_core.Face object at 0x000001F621A88630>, <topologic_core.Face object at 0x000001F621A8AAF0>]
Apertures:  [<topologic_core.Face object at 0x000001F63C6B90F0>, <topologic_core.Face object at 0x000001F67FD7FFB0>, <topologic_core.Face object at 0x000001F

In [24]:
geometryWithApertures = Topology.AddApertures(cc, apertures, subTopologyType="face")  

In [25]:
g2 = Graph.ByTopology(geometryWithApertures, direct=False, viaSharedApertures=True, toExteriorApertures=True) 

In [26]:
vertices = Graph.Vertices(g2)
for v in vertices:
    d = Dictionary.ByKeysValues(["size", "color"], [18, "red"])
    v = Topology.SetDictionary(v, d)
edges = Graph.Edges(g2)
for e in edges:
    d = Topology.Dictionary(e)
    d = Dictionary.SetValuesAtKeys(d, ["width", "color"], [4, "black"]) #to expand the dictionary and not overwrite it and then put it back
    e = Topology.SetDictionary(e, d)

In [27]:
Topology.Show(cc, apertures, g2,
              vertexSizeKey="size",
              vertexColorKey="color",
              edgeWidthKey="width",
              edgeColorKey="color",
              faceOpacity=0.3,
              backgroundColor="white",
              width=500,
              height=500,
              renderer=renderer)